In [ ]:
import logging

import teehr
from teehr.evaluation.evaluation import RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.fetching.utils import format_nwm_configuration_metadata
from teehr import Variable

logger = logging.getLogger()

In [ ]:
spark = create_spark_session(
    aws_profile="admin-user"
)
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

In [ ]:
nwm_version = "nwm30"

secondary_nwm_config_names_to_add = [
    "analysis_assim_no_da",
    "analysis_assim_alaska_no_da",
    "analysis_assim_extend_no_da",
    "analysis_assim_extend_alaska_no_da",
    "analysis_assim_hawaii_no_da",
    "analysis_assim_puertorico_no_da",
    "short_range_alaska",
    "short_range_hawaii",
    "short_range_puertorico",
    "medium_range_alaska_mem1",
    "medium_range_blend",
    "medium_range_blend_alas",
    "forcing_short_range",
    "forcing_short_range_alaska",
    "forcing_short_range_hawaii",
    "forcing_short_range_puertori",
    "forcing_medium_range",
    "forcing_medium_range_alaska",
    "forcing_medium_range_blend",
    "forcing_medium_range_blend_alaska"
]

primary_nwm_config_names_to_add = [
    "forcing_analysis_assim",
    "forcing_analysis_assim_alaska",
    "forcing_analysis_assim_extend",
    "forcing_analysis_assim_extend_alaska",
    "forcing_analysis_assim_hawaii",
    "forcing_analysis_assim_puertorico",
]

In [ ]:
%%time
for nwm_configuration in secondary_nwm_config_names_to_add:
    ev_config = format_nwm_configuration_metadata(
        nwm_config_name=nwm_configuration,
        nwm_version=nwm_version
    )
    ev.configurations.add(
        configuration=[
            teehr.Configuration(
                name=ev_config["name"],
                timeseries_type="secondary",
                description=ev_config["description"],
            )
        ]
    )

In [ ]:
%%time
for nwm_configuration in primary_nwm_config_names_to_add:
    ev_config = format_nwm_configuration_metadata(
        nwm_config_name=nwm_configuration,
        nwm_version=nwm_version
    )
    ev.configurations.add(
        configuration=[
            teehr.Configuration(
                name=ev_config["name"],
                timeseries_type="primary",
                description=ev_config["description"],
            )
        ]
    )

In [ ]:
ev.configurations.to_sdf().show(n=50, truncate=False)

In [ ]:
ev.variables.add(
    Variable(
        name="streamflow_15min_inst",
        long_name="15-minute Instantaneous Streamflow",
    )
)